# Projeto SQL — Análise de Serviço de Livros

Durante a pandemia, o hábito de leitura cresceu e novas startups surgiram para atender leitores. Recebemos o banco de dados de um serviço concorrente — livros, autores, editoras, classificações e avaliações — para extrair insights que apoiem a proposta de um novo produto.

Este notebook responde a cinco perguntas de negócio via SQL, usando pandas apenas para executar consultas e exibir resultados.

## Passo 1 — Objetivos do estudo

Entender o catálogo e o comportamento dos usuários para embasar decisões de produto:

1. Dimensionar o catálogo moderno (livros após 01/01/2000).
2. Medir engajamento por livro (nº de avaliações e classificação média).
3. Identificar a editora mais relevante em livros >50 páginas.
4. Encontrar autores mais bem avaliados (massa crítica de ≥50 classificações).
5. Medir o engajamento dos usuários mais ativos (>50 livros classificados).

In [1]:
# importa bibliotecas
import pandas as pd
from sqlalchemy import create_engine, text

# configura acesso ao DB
db_config = {
 'user': 'practicum_student', # username
 'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7', # password
 'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
 'port': 5432, # connection port
 'db': 'data-analyst-final-project-db' # the name of the database
 }
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
db_config['pwd'],
db_config['host'],
db_config['port'],
db_config['db'])
engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [2]:
# define função para executar as 'queries' pelo notebook todo
def run_query(query):
    '''Executa uma consulta SQL e retorna o resultado como DataFrame.'''
    with engine.connect() as conn:
        return pd.read_sql(text(query), con=conn)

## Passo 2 — Conexão e exploração das tabelas

Amostra das cinco tabelas para confirmar tipos, chaves e granularidade antes de consultar.

In [3]:
for table in ['books', 'authors', 'publishers', 'ratings', 'reviews']:
    display(run_query(f'SELECT * FROM {table} LIMIT 5'))

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


## Passo 3 — Livros lançados após 01/01/2000

**Tarefa:** contar quantos livros foram publicados depois de 01/01/2000.
**Abordagem:** `COUNT` sobre `books`, filtrando `publication_date` com `WHERE`.

In [4]:
query = '''
    SELECT COUNT(book_id) AS qtd_livros
    FROM books WHERE publication_date > '2000-01-01'
    '''
run_query(query)

,qtd_livros
0,819


**Resultado e conclusão:** 819 livros. O catálogo é fortemente concentrado em publicações recentes (819 de 1000 livros são 82%) — a base atende bem o leitor de lançamentos, não o de clássicos. Interpretei "depois de" como `>` (exclusivo); ver Diário de Decisões.

## Passo 4 — Avaliações e classificação média por livro

**Tarefa:** para cada livro, o número de avaliações (reviews) e a classificação média (ratings).
**Abordagem:** `books` como base, `LEFT JOIN` com `reviews` e `ratings`. `COUNT(DISTINCT review_id)` evita a inflação do produto cartesiano; `AVG(rating)` não sofre com a duplicação uniforme.

In [5]:
query = '''
    SELECT
        b.book_id,
        b.title,
        COUNT(DISTINCT rev.review_id) AS qtd_avaliacoes,
        AVG(rat.rating) AS classificacao_media
    FROM books AS b
    LEFT JOIN reviews AS rev ON b.book_id = rev.book_id
    LEFT JOIN ratings AS rat ON b.book_id = rat.book_id
    GROUP BY b.book_id, b.title
    '''
run_query(query)

,book_id,title,qtd_avaliacoes,classificacao_media
0,1,'Salem's Lot,2,3.666667
1,2,1 000 Places to See Before You Die,1,2.500000
2,3,13 Little Blue Envelopes (Little Blue Envelope...,3,4.666667
3,4,1491: New Revelations of the Americas Before C...,2,4.500000
4,5,1776,4,4.000000
...,...,...,...,...
995,996,Wyrd Sisters (Discworld #6; Witches #2),3,3.666667
996,997,Xenocide (Ender's Saga #3),3,3.400000
997,998,Year of Wonders,4,3.200000
998,999,You Suck (A Love Story #2),2,4.500000


In [6]:
resultado = run_query(query)
print('NULL por coluna:')
print(resultado.isna().sum())
print('\nLivros com zero avaliações:', (resultado['qtd_avaliacoes'] == 0).sum())

NULL por coluna:
book_id                0
title                  0
qtd_avaliacoes         0
classificacao_media    0
dtype: int64

Livros com zero avaliações: 6


**Resultado e conclusão:** 1000 livros (catálogo completo). 6 sem nenhuma avaliação em texto — mas todos os 1000 têm nota (zero NULL em `classificacao_media`). Ou seja: classificar e escrever review são engajamentos independentes, e o texto é o canal mais escasso. Retê-los só foi possível pelo `LEFT JOIN`.

## Passo 5 — Editora com mais livros (>50 páginas)

**Tarefa:** identificar a editora que lançou o maior número de livros com mais de 50 páginas, excluindo brochuras e publicações curtas.
**Abordagem:** `JOIN` entre `books` e `publishers` por `publisher_id`, filtrando `num_pages > 50` com `WHERE`. Agrupo por editora, ordeno pela contagem em ordem decrescente e pego a primeira com `LIMIT 1`.

In [7]:
query = '''
    SELECT
        p.publisher,
        COUNT(b.book_id) AS qtd_livros
    FROM books AS b
    JOIN publishers AS p ON b.publisher_id = p.publisher_id
    WHERE b.num_pages > 50
    GROUP BY p.publisher_id, b.publisher_id
    ORDER BY qtd_livros DESC
    LIMIT 3
'''
run_query(query)

,publisher,qtd_livros
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25


**Resultado e conclusão:** Penguin Books lidera com 42 livros acima de 50 páginas. O resultado é coerente com o porte da editora no mercado — uma editora grande e generalista no topo valida a consulta. Para o produto, sinaliza que parcerias ou curadoria com grandes casas editoriais alcançam a maior fatia do catálogo relevante.

## Passo 6 — Autor com maior média de classificação (≥50 classificações)

**Tarefa:** identificar o autor com a média de classificação mais alta, considerando apenas livros com pelo menos 50 classificações.
**Abordagem:** subconsulta em duas camadas. A interna calcula a média e a contagem de classificações **por livro**, filtrando com `HAVING COUNT >= 50`. A externa trata esse resultado como tabela, junta com `authors` e tira a média das médias **por autor**.

Antes de agregar por autor, isolo os livros com pelo menos 50 classificações — a base da subconsulta. São eles que sobrevivem ao piso e alimentam o cálculo por autor logo abaixo.

In [8]:
# Livros que passaram no corte de 50+ classificações (base da subconsulta abaixo)
query = '''
    SELECT
        b.book_id,
        b.author_id,
        AVG(rat.rating) AS media_livro,
        COUNT(rat.rating_id) AS qtd_classificacoes
    FROM books AS b
    JOIN ratings AS rat ON b.book_id = rat.book_id
    GROUP BY b.book_id, b.author_id
    HAVING COUNT(rat.rating_id) >= 50
    ORDER BY qtd_classificacoes DESC
'''
run_query(query)

,book_id,author_id,media_livro,qtd_classificacoes
0,948,554,3.662500,160
1,750,240,4.125000,88
2,673,235,3.825581,86
3,75,106,3.678571,84
4,302,236,4.414634,82
5,299,236,4.287500,80
6,301,236,4.186667,75
7,79,195,3.729730,74
8,722,240,4.391892,74
9,300,236,4.246575,73


In [9]:
query = '''
    SELECT
        au.author,
        AVG(sub.media_livro) AS media_autor
    FROM (
        SELECT
            b.author_id,
            b.book_id,
            AVG(rat.rating) AS media_livro,
            COUNT(rat.rating_id) AS qtd_classificacoes
        FROM books AS b
        JOIN ratings AS rat ON b.book_id = rat.book_id
        GROUP BY b.book_id, b.author_id
        HAVING COUNT(rat.rating_id) >= 50
    ) AS sub
    JOIN authors AS au ON au.author_id = sub.author_id
    GROUP BY au.author
    ORDER BY media_autor DESC
    LIMIT 3
'''
run_query(query)

,author,media_autor
0,J.K. Rowling/Mary GrandPré,4.283844
1,Markus Zusak/Cao Xuân Việt Khương,4.264151
2,J.R.R. Tolkien,4.258446


**Resultado e conclusão:** J.K. Rowling/Mary GrandPré lidera com média ~4,28. Apenas 18 dos 1000 livros passaram no corte de 50+ classificações — a base tem cauda longa, e o piso protege o ranking contra livros com poucas notas infladas. O campo de autor concatena colaboradores (autora/ilustradora), característica da tabela de origem.

## Passo 7 — Média de avaliações entre usuários muito ativos (>50 livros)

**Tarefa:** encontrar o número médio de avaliações (reviews) entre usuários que classificaram mais de 50 livros.
**Abordagem:** três camadas. A mais interna isola os usuários com mais de 50 livros classificados (`HAVING COUNT(DISTINCT book_id) > 50`). A intermediária conta quantas reviews cada um desses usuários escreveu. A externa tira a média dessas contagens.

In [10]:
query = '''
    SELECT AVG(sub.qtd_avaliacoes) AS media_avaliacoes
    FROM (
        SELECT
            rev.username,
            COUNT(rev.review_id) AS qtd_avaliacoes
        FROM reviews AS rev
        WHERE rev.username IN (
            SELECT rat.username
            FROM ratings AS rat
            GROUP BY rat.username
            HAVING COUNT(DISTINCT rat.book_id) > 50
        )
        GROUP BY rev.username
    ) AS sub
'''
run_query(query)

,media_avaliacoes
0,24.333333


**Resultado e conclusão:** apenas 6 usuários classificaram mais de 50 livros — os super-usuários da base. Todos os seis também escrevem reviews, com média de ~24,3 avaliações cada. Esse grupo é hiperengajado nos dois canais (nota e texto), o perfil de maior valor para o produto: gera dado estruturado e conteúdo simultaneamente.

## Diário de Decisões Analíticas

Registro das escolhas metodológicas sob meu julgamento. Passos sem entrada não exigiram decisão — foram execução direta do enunciado.

**Passo 3 — [autoral]** Interpretei "depois de 1 de janeiro de 2000" como `>` (exclusivo). Livros de 2000-01-01 ficam fora. Resultado: 819.

**Passo 4 — [autoral]** Optei por `LEFT JOIN` em vez de `INNER`. A verificação confirmou 6 livros sem avaliação em texto — com `INNER JOIN` sobre reviews eles cairiam e o resultado seria 994, não 1000. O `LEFT` foi decisão necessária para preservar o catálogo completo.

**Passo 6 — [autoral]** Calculei "média das médias por livro" (cada livro com peso igual), leitura direta do enunciado. A alternativa — média ponderada por número de classificações — daria resultado diferente, mas priorizei a interpretação literal da tarefa.

**Passo 7 — [autoral]** Usei `COUNT(DISTINCT rat.book_id)` em vez de `COUNT(rating_id)` para contar livros, não classificações. Se o sistema permitir múltiplas classificações do mesmo livro por um usuário (ex.: edições com novo id), o distinct garante fidelidade ao enunciado ("mais de 50 livros"). Verifiquei que os 6 usuários têm ao menos uma review, então nenhum ficou fora da média por ausência no canal de texto.```

## Passo 8 — Conclusões gerais

O catálogo é moderno e amplo: 819 livros pós-2000, num acervo de 1000. O engajamento, porém, é profundamente desigual — a maioria dos livros tem poucas avaliações, e apenas 18 acumularam massa crítica de 50+ classificações. Entre autores com essa massa, J.K. Rowling/Mary GrandPré lidera (~4,28). No lado da oferta, Penguin Books domina o catálogo relevante (>50 páginas) com 42 títulos.

O achado mais acionável está nos usuários: só 6 pessoas classificaram mais de 50 livros, e esse mesmo grupo escreve ~24 reviews em média. A base é sustentada por uma pequena elite hiperengajada. Para o novo produto, isso sugere duas frentes: (1) cultivar e reter esses super-usuários, que geram a maior parte do sinal de dados; (2) reduzir a fricção para o usuário mediano avaliar, ampliando a base de engajamento além do núcleo atual.